---
description: Orchestrate audit, ingestion, indexing, retrieval, and cited answer generation
output-file: application.layout_rag.html
title: Layout-aware OCR RAG application
---

In [ ]:
# | default_exp application.layout_rag

# Layout-aware OCR RAG application

This notebook is the application boundary that connects the repository's
layout-aware OCR modules. It audits and optionally repairs one evidence
bundle, constructs canonical hierarchy and typed retrieval records, indexes
them, retrieves exact evidence, and assembles page/region/bbox citations.

The online query path never reopens or repairs source files. The optional
answer generator receives exact cited evidence rather than summaries alone.
SQLite, Chroma, embedding, visual, reranking, repair, and generation resources
remain explicit dependencies owned by the caller.

In [ ]:
# | export
from __future__ import annotations

import inspect
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Awaitable, Callable, Mapping, Sequence

from ribosome.preprocessing.ocr.audit_repair import (
    OCRFileQualityReport,
    OCRFixAction,
    OCRRepairHandler,
    OCRRepairLoopResult,
    audit_layout_ocr_file,
    execute_ocr_repairs_until_stable,
)
from ribosome.preprocessing.ocr.layout_bundle import (
    LayoutIngestionResult,
    ingest_layout_bundle,
)
from ribosome.retrieval.layout_rag import (
    CitationAssembler,
    DenseCandidateIndex,
    EmbeddingProvider,
    Evidence,
    HybridIndexer,
    HybridRetriever,
    LayoutRAGConfig,
    Reranker,
    SQLiteRetrievalRecordStore,
    VisualRetriever,
)

## Application results and extension contracts

In [ ]:
# | export
class OCRAuditBlockedError(ValueError):
    """Raised before indexing when the canonical OCR evidence is not clean."""

    def __init__(
        self,
        report: OCRFileQualityReport,
        repair: OCRRepairLoopResult | None = None,
    ):
        self.report = report
        self.repair = repair
        finding_count = len(report.file_reasons) + len(report.region_issues)
        repair_suffix = (
            f" after repair stopped with {repair.stop_reason!r}"
            if repair is not None
            else ""
        )
        super().__init__(
            f"OCR audit blocked indexing for {report.sidecar_path}: "
            f"{finding_count} finding(s){repair_suffix}"
        )


@dataclass(frozen=True)
class LayoutRAGIndexResult:
    """Final audit, optional repair run, canonical ingestion, and store counts."""

    audit: OCRFileQualityReport
    repair: OCRRepairLoopResult | None
    ingestion: LayoutIngestionResult
    counts: Mapping[str, int]

    @property
    def document_id(self) -> str:
        return self.ingestion.document.document_id


@dataclass(frozen=True)
class LayoutRAGQueryResult:
    """Ordered retrieval evidence and its machine-readable citations."""

    query: str
    evidence: tuple[Evidence, ...]
    citations: tuple[dict[str, Any], ...]

    @property
    def answerable(self) -> bool:
        return bool(self.evidence)

    @property
    def context(self) -> str:
        """Exact evidence blocks labelled for a downstream generator."""
        return "\n\n".join(
            f"[E{position}] {item.citation}\n{item.exact_text}"
            for position, item in enumerate(self.evidence, start=1)
        )


@dataclass(frozen=True)
class LayoutRAGAnswerRequest:
    """The complete, exact-evidence request passed to an answer generator."""

    query: str
    evidence: tuple[Evidence, ...]
    citations: tuple[dict[str, Any], ...]
    context: str


AnswerGenerator = Callable[[LayoutRAGAnswerRequest], str | Awaitable[str]]


@dataclass(frozen=True)
class LayoutRAGAnswer:
    """Generated text plus the evidence and citations that ground it."""

    query_result: LayoutRAGQueryResult
    text: str | None

    @property
    def abstained(self) -> bool:
        return self.text is None

## Audit, ingest, index, retrieve, and answer

In [ ]:
# | export
def _validated_vector(values: Sequence[float], *, label: str) -> list[float]:
    try:
        vector = [float(value) for value in values]
    except (TypeError, ValueError) as error:
        raise ValueError(f"{label} embedding must be a numeric sequence") from error
    if not vector:
        raise ValueError(f"{label} embedding must not be empty")
    if any(not math.isfinite(value) for value in vector):
        raise ValueError(f"{label} embedding contains a non-finite value")
    return vector


async def _maybe_await(value: Any) -> Any:
    return await value if inspect.isawaitable(value) else value


class LayoutRAGApplication:
    """Application service joining canonical preprocessing and hybrid retrieval."""

    def __init__(
        self,
        store: SQLiteRetrievalRecordStore,
        *,
        dense_index: DenseCandidateIndex | None = None,
        embedder: EmbeddingProvider | None = None,
        reranker: Reranker | None = None,
        visual_retriever: VisualRetriever | None = None,
        answer_generator: AnswerGenerator | None = None,
        config: LayoutRAGConfig | None = None,
    ):
        self.store = store
        self.dense_index = dense_index
        self.embedder = embedder
        self.answer_generator = answer_generator
        self.config = config or LayoutRAGConfig()
        dense_version = getattr(dense_index, "pipeline_version", self.config.pipeline_version)
        if dense_index is not None and dense_version != self.config.pipeline_version:
            raise ValueError(
                "dense index and retrieval configuration pipeline versions differ"
            )
        self.indexer = HybridIndexer(store, dense_index)
        self.retriever = HybridRetriever(
            store,
            dense_index=dense_index,
            reranker=reranker,
            visual_retriever=visual_retriever,
            config=self.config,
        )

    async def index_bundle(
        self,
        layout_path: str | Path,
        *,
        markdown_path: str | Path | None = None,
        pdf_path: str | Path | None = None,
        pdf_roots: Sequence[str | Path] = (),
        reconcile_native_text: bool = True,
        strict: bool = True,
        repair_handlers: Mapping[OCRFixAction, OCRRepairHandler] | None = None,
        max_repair_rounds: int = 3,
        child_embeddings: Sequence[Sequence[float]] | None = None,
        parent_embeddings: Sequence[Sequence[float]] | None = None,
    ) -> LayoutRAGIndexResult:
        """Gate, optionally repair, ingest, and atomically index one bundle."""
        selected_layout = Path(layout_path).expanduser().resolve()
        audit = audit_layout_ocr_file(selected_layout)
        repair: OCRRepairLoopResult | None = None
        if audit.needs_repair:
            if not repair_handlers:
                raise OCRAuditBlockedError(audit)
            repair = await execute_ocr_repairs_until_stable(
                audit.sidecar_path,
                repair_handlers,
                max_rounds=max_repair_rounds,
                show_progress=False,
                stream_reports=False,
            )
            audit = audit_layout_ocr_file(audit.sidecar_path)
            if audit.needs_repair:
                raise OCRAuditBlockedError(audit, repair)

        ingestion = ingest_layout_bundle(
            audit.sidecar_path,
            markdown_path=markdown_path,
            pdf_path=pdf_path,
            pdf_roots=pdf_roots,
            pipeline_version=self.config.pipeline_version,
            reconcile_native_text=reconcile_native_text,
            strict=strict,
        )
        await self.indexer.index(
            ingestion,
            embedder=self.embedder,
            child_embeddings=child_embeddings,
            parent_embeddings=parent_embeddings,
        )
        counts = self.store.counts(
            ingestion.document.document_id,
            pipeline_version=self.config.pipeline_version,
        )
        return LayoutRAGIndexResult(audit, repair, ingestion, counts)

    async def _embed_query(self, query: str) -> list[float]:
        if self.embedder is None:
            raise ValueError(
                "dense retrieval requires an embedder or an explicit query embedding"
            )
        embeddings = await _maybe_await(self.embedder([query]))
        if not isinstance(embeddings, Sequence) or len(embeddings) != 1:
            raise ValueError("embedding provider must return exactly one query embedding")
        return _validated_vector(embeddings[0], label="query")

    async def query(
        self,
        query: str,
        *,
        query_embedding: Sequence[float] | None = None,
        limit: int | None = None,
    ) -> LayoutRAGQueryResult:
        """Retrieve exact evidence and ordered citation payloads."""
        if not query.strip():
            return LayoutRAGQueryResult(query, (), ())
        if query_embedding is not None and self.dense_index is None:
            raise ValueError("a query embedding was supplied without a dense index")
        vector = (
            _validated_vector(query_embedding, label="query")
            if query_embedding is not None
            else await self._embed_query(query)
            if self.dense_index is not None
            else None
        )
        evidence = self.retriever.retrieve_evidence(
            query,
            query_embedding=vector,
            limit=limit,
        )
        citations = tuple(
            CitationAssembler.citation_payload(item) for item in evidence
        )
        return LayoutRAGQueryResult(query, evidence, citations)

    async def answer(
        self,
        query: str,
        *,
        query_embedding: Sequence[float] | None = None,
        limit: int | None = None,
    ) -> LayoutRAGAnswer:
        """Generate from exact cited context, or abstain when retrieval is empty."""
        query_result = await self.query(
            query, query_embedding=query_embedding, limit=limit
        )
        if not query_result.answerable:
            return LayoutRAGAnswer(query_result, None)
        if self.answer_generator is None:
            raise RuntimeError("answer generation requires an answer_generator")
        request = LayoutRAGAnswerRequest(
            query=query_result.query,
            evidence=query_result.evidence,
            citations=query_result.citations,
            context=query_result.context,
        )
        generated = await _maybe_await(self.answer_generator(request))
        if not isinstance(generated, str):
            raise TypeError("answer_generator must return a string")
        text = generated.strip() or None
        return LayoutRAGAnswer(query_result, text)

## Minimal lexical application

Lexical indexing and retrieval need no embedding service. A synchronous
script can wrap these calls with `asyncio.run`; Jupyter supports top-level
`await`.

In [ ]:
# | eval: false
with SQLiteRetrievalRecordStore("layout-rag.sqlite3") as store:
    app = LayoutRAGApplication(store)
    indexed = await app.index_bundle(
        "document.layout.json",
        markdown_path="document.md",
        pdf_path="document.pdf",
    )
    result = await app.query("MOVJ P[1] V=10 ACC=100 CNT=100")
    for evidence, citation in zip(result.evidence, result.citations):
        print(evidence.citation, evidence.exact_text)
        print(citation["bboxes"])

## Optional dense and generation adapters

Pass a `ChromaDenseIndex` and one embedding provider to use the same vector
space for child, parent, and query embeddings. Visual retrieval, reranking,
repair handlers, and answer generation are injected explicitly because their
models and service policies are deployment choices. The answer-generator
contract receives `[E1]`, `[E2]`, ... exact evidence blocks and the associated
machine-readable page/region/bbox citation payloads.